## Mutation profile preprocessing and PERMANOVA

##### This notebook:
##### 1. Loads and filters metagenomic and clone WGS variant data.
##### 2. Curates high-confidence non-synonymous variants.
##### 3. Builds community-level mutation presence/absence matrices.
##### 4. Runs:
#####    - PERMANOVA on mutation profiles,
#####    - Pairwise PERMANOVA post-hoc tests,
#####    - Tests for dispersion homogeneity,
#####    - Tests for differences in total mutation counts.
##### 5. Prints out all statistical outputs

#### 1. Load packages and set up output directory
##### Load required libraries and ensure the stats output directory exists

In [1]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(pairwiseAdonis)
  library(reshape2)
  library(vegan)
  library(writexl)
})

#### 2. Read input variant tables and select relevant columns
##### Import metagenome and WGS variant calls and keep only required fields

In [2]:
df_metagen <- read_tsv("../data/metagenome_variant_calls.tsv",
                       show_col_types = FALSE)
df_wgs     <- read_tsv("../data/clone_wgs_variant_calls.tsv",
                       show_col_types = FALSE)

df_metagen <- df_metagen[, c("Sample","HAMBI", "POS","CHROM", "GENE",
                             "product1", "gene_name1", "AF",
                             "EFFECT", "TYPE", "DP", "HGVS_P")]

df_wgs <- df_wgs[, c("Sample", "POS","CHROM", "GENE",
                     "product1", "gene_name1", "AF",
                     "EFFECT", "TYPE", "DP", "HGVS_P")]

#### 3. Define unique variant IDs and mark shared variants
##### Create uniqvar IDs and flag whether variants are present in both datasets

In [3]:
df_metagen$uniqvar <- paste(df_metagen$CHROM, df_metagen$GENE, df_metagen$HGVS_P,
                            sep = "_")
df_wgs$uniqvar     <- paste(df_wgs$CHROM,     df_wgs$GENE,     df_wgs$HGVS_P,
                            sep = "_")

df_metagen$evo_presence <- ifelse(df_metagen$uniqvar %in% unique(df_wgs$uniqvar),
                                  "yes", "no")
df_wgs$evo_presence     <- ifelse(df_wgs$uniqvar %in% unique(df_metagen$uniqvar),
                                  "yes", "no")

#### 4. Remove CC-only genes not observed in WGS
##### Filter out genes seen only in the CC sample and absent from the WGS data

In [4]:
CC       <- df_metagen[df_metagen$Sample == "CC", ]
CC_genes <- unique(CC$GENE)
CC_genes <- CC_genes[!(CC_genes %in% unique(df_wgs$GENE))]

df_metagen <- df_metagen[!(df_metagen$GENE %in% CC_genes), ]

#### 5. Restrict WGS to confident variants and harmonize metagenome genes
##### Apply DP/AF filters to WGS and remove metagenome genes lacking confident WGS support

In [5]:
df_wgs2 <- df_wgs[df_wgs$DP > 20 & df_wgs$AF > 0.8, ]

evo_gene_to_keep   <- df_wgs2$GENE
evo_gene_to_remove <- df_wgs$GENE[!(df_wgs$GENE %in% evo_gene_to_keep)]

df_metagen2 <- df_metagen[!(df_metagen$GENE %in% evo_gene_to_remove), ]

#### 6. Keep variants with sufficient metagenome coverage or present in WGS
##### Compute maximum AF per variant and filter by AF > 0.3 or WGS presence

In [6]:
maxfreqs <- aggregate(data = df_metagen2[df_metagen2$DP > 20, ],
                      AF ~ uniqvar, FUN = max)
maxfreqs <- maxfreqs[order(maxfreqs$AF, decreasing = TRUE), ]

maxfreq_30_or_more <- maxfreqs$uniqvar[maxfreqs$AF > 0.3]

df_metagen3 <- df_metagen2[
  df_metagen2$uniqvar %in% maxfreq_30_or_more |
    df_metagen2$uniqvar %in% unique(df_wgs2$uniqvar),
]

#### 7. Annotate Evo, replicate, and primer pulse treatments
##### Decode treatment, replicate, and primer pulse from sample names

In [7]:
# Evo treatment
df_metagen3$Evo <- NA
df_metagen3$Evo <- ifelse(grepl("^a-", df_metagen3$Sample),
                          "All ancestral", df_metagen3$Evo)
df_metagen3$Evo <- ifelse(grepl("^d-", df_metagen3$Sample),
                          "Pre-exposed 1972", df_metagen3$Evo)
df_metagen3$Evo <- ifelse(grepl("^e-", df_metagen3$Sample),
                          "Pre-exposed 1977", df_metagen3$Evo)
df_metagen3$Evo <- ifelse(grepl("^f-", df_metagen3$Sample),
                          "All pre-exposed", df_metagen3$Evo)
table(df_metagen3$Evo)

# Replicate
df_metagen3$Replicate <- NA
df_metagen3$Replicate <- ifelse(grepl("-1-", df_metagen3$Sample),
                                1, df_metagen3$Replicate)
df_metagen3$Replicate <- ifelse(grepl("-2-", df_metagen3$Sample),
                                2, df_metagen3$Replicate)
df_metagen3$Replicate <- ifelse(grepl("-3-", df_metagen3$Sample),
                                3, df_metagen3$Replicate)
df_metagen3$Replicate <- ifelse(grepl("-4-", df_metagen3$Sample),
                                4, df_metagen3$Replicate)
table(df_metagen3$Replicate)

# Primer pulse treatment
df_metagen3$Primer_pulse <- NA
df_metagen3$Primer_pulse <- ifelse(grepl("-t1-", df_metagen3$Sample),
                                   5,   df_metagen3$Primer_pulse)
df_metagen3$Primer_pulse <- ifelse(grepl("-t2-", df_metagen3$Sample),
                                   50,  df_metagen3$Primer_pulse)
df_metagen3$Primer_pulse <- ifelse(grepl("-t3-", df_metagen3$Sample),
                                   500, df_metagen3$Primer_pulse)
df_metagen3$Primer_pulse <- ifelse(grepl("-t4-", df_metagen3$Sample),
                                   0,   df_metagen3$Primer_pulse)
table(df_metagen3$Primer_pulse)


   All ancestral  All pre-exposed Pre-exposed 1972 Pre-exposed 1977 
             282              380              373              317 


  1   2   3   4 
345 327 349 331 


  0   5  50 500 
328 361 324 339 

#### 8. Special handling of near-duplicate variants
##### Manually set `evo_presence = "yes"` for gene **H1972_03930**, which is always present at a nearby position (bioinformatic mis-assignment).

In [8]:
df_wgs2$evo_presence[df_wgs2$GENE == "H1972_03930"]       <- "yes"
df_metagen3$evo_presence[df_metagen3$GENE == "H1972_03930"] <- "yes"

df_metagen3$COUNT <- 1L

#### 9. Remove duplicated variants within the same population
##### Drop variants that appear more than once per Evo × Primer_pulse × Replicate

In [9]:
abund_targ <- aggregate(
  COUNT ~ uniqvar + Evo + Primer_pulse + Replicate,
  data = df_metagen3,
  FUN  = sum
)
abund_targ <- abund_targ[order(abund_targ$COUNT, decreasing = TRUE), ]

abund_targ_uniqvars <- abund_targ$uniqvar[abund_targ$COUNT > 1]

df_metagen4 <- df_metagen3[!(df_metagen3$uniqvar %in% abund_targ_uniqvars), ]
df_wgs2     <- df_wgs2[!(df_wgs2$uniqvar %in% abund_targ_uniqvars), ]

df_metagen4$COUNT <- 1
abund_targ <- aggregate(
  data = df_metagen4,
  COUNT ~ uniqvar + Evo + Primer_pulse + Replicate,
  FUN   = sum
)
abund_targ <- abund_targ[order(abund_targ$COUNT, decreasing = TRUE), ]

abund_targ_uniqvars <- abund_targ$uniqvar[abund_targ$COUNT > 1]

df_metagen4 <- df_metagen4[!(df_metagen4$uniqvar %in% abund_targ_uniqvars), ]
df_wgs2     <- df_wgs2[!(df_wgs2$uniqvar %in% abund_targ_uniqvars), ]

#### 10. Remove suspiciously ubiquitous variants across populations
##### Some variants are present in a very large number of independent populations, which is suspicious for true de novo adaptation signals and more likely reflects mapping / reference issues.
##### We:
##### - count in how many **independent populations** (`Evo × Primer_pulse × Replicate`) each `uniqvar` appears, using the original ABC logic; and
##### - drop variants that occur in **> 40** populations from both:
#####   - the metagenomic dataset (`df_metagen4`) and
#####   - the clone WGS dataset (`df_wgs2`).

In [10]:
df_popanalys <- aggregate(
  data = df_metagen3,
  AF ~ uniqvar + Evo + Primer_pulse + Replicate,
  FUN = sum
)
df_popanalys$COUNT <- 1

df_popanalys2 <- aggregate(
  data  = df_popanalys,
  COUNT ~ uniqvar,
  FUN   = sum
)
df_popanalys2 <- df_popanalys2[order(df_popanalys2$COUNT, decreasing = TRUE), ]

susp_anc_var <- df_popanalys2$uniqvar[df_popanalys2$COUNT > 40]

df_metagen5 <- df_metagen4[!(df_metagen4$uniqvar %in% susp_anc_var), ]
df_wgs2     <- df_wgs2[!(df_wgs2$uniqvar %in% susp_anc_var), ]

unique(df_metagen5$EFFECT)

[1] "missense_variant"                          
 [2] "intergenic_region"                         
 [3] "synonymous_variant"                        
 [4] "frameshift_variant&stop_gained"            
 [5] "frameshift_variant"                        
 [6] "stop_gained&conservative_inframe_insertion"
 [7] "disruptive_inframe_insertion"              
 [8] "stop_gained"                               
 [9] "stop_gained&disruptive_inframe_insertion"  
[10] "disruptive_inframe_deletion"               
[11] "non_coding_transcript_variant"             
[12] "stop_lost&splice_region_variant"           
[13] "conservative_inframe_insertion"

#### 11. Keep non-synonymous / impactful variants **and** gene-level recurrence
##### This step has two layers: 
##### 1. **Functional filter:** drop low-impact / synonymous variant annotations from both metagenomes and WGS.
##### 2. **Recurrent gene filter (new explicit step):**
#####    - define an *independent population* as a unique combination of `Evo`, `Primer_pulse`, and `Replicate`;
#####    - a gene is **recurrent** if it carries at least one non-synonymous / impactful mutation in **≥ 2 independent populations**;
#####   - keep only variants in such recurrent genes in the metagenomic dataset.
##### The resulting objects:
##### - `df_wgs3`: curated WGS non-synonymous variants (no recurrence filter).
##### - `df_metagen6`: curated metagenomic non-synonymous variants, restricted to **recurrently mutated genes**.

In [11]:
df_metagen6 <- df_metagen5[!(df_metagen5$EFFECT %in% c(
  "intergenic_region",
  "synonymous_variant",
  "conservative_inframe_insertion",
  "non_coding_transcript_variant",
  "initiator_codon_variant",
  "conservative_inframe_deletion",
  "conservative_inframe_insertion&splice_region_variant"
)), ]

df_wgs3 <- df_wgs2[!(df_wgs2$EFFECT %in% c(
  "intergenic_region",
  "synonymous_variant",
  "conservative_inframe_insertion",
  "non_coding_transcript_variant",
  "initiator_codon_variant",
  "conservative_inframe_deletion",
  "conservative_inframe_insertion&splice_region_variant"
)), ]

#### 12. Curate WGS mutation table and write to disk
##### Standardize sample IDs, build a Variant ID, and export curated WGS non-synonymous mutations

In [12]:
df_wgs3$HAMBI      <- gsub("_S.*", "", df_wgs3$Sample)
df_wgs3$Variant    <- paste(df_wgs3$GENE, df_wgs3$gene_name1, df_wgs3$HGVS_P, sep = "_")
df_wgs3$detectable <- ifelse(
  df_wgs3$Sample == "Evo_1972_S43",
  "Mutation detectable (abundant species)",
  "Uncertain detectability (low-abundance species)"
)

df_wgs3$Sample <- gsub("Evo",   "HAMBI", df_wgs3$Sample)
df_wgs3$Sample <- gsub("_S.*", "",      df_wgs3$Sample)

write.table(df_wgs3,
            file   = "../data/curated_wgs_nonsyn_mutations.txt",
            quote  = FALSE,
            sep    = "\t")

#### 13. Aggregate metagenomic non-synonymous variants for community-level counts
##### - Combine gene ID and name into `Variantgene`.
##### - Aggregate hits per **HAMBI × Variantgene × Primer_pulse × Evo × evo_presence × Replicate**.

In [13]:
df_metagen6$Variantgene <- paste(df_metagen6$GENE, df_metagen6$gene_name1, sep = "_")

df_metagen6$COUNT <- 1
highfreq_aggr <- aggregate(
  COUNT ~ HAMBI * Variantgene * Primer_pulse * Evo * evo_presence * Replicate,
  FUN   = sum,
  data  = df_metagen6
)
highfreq_aggr$COUNT <- 1

highfreq_aggr2 <- aggregate(
  COUNT ~ HAMBI * Variantgene * Primer_pulse * Evo * evo_presence,
  FUN   = sum,
  data  = highfreq_aggr
)
highfreq_aggr2 <- highfreq_aggr2[order(highfreq_aggr2$COUNT, decreasing = TRUE), ]

highfreq_aggr2$Primer_pulse <- factor(
  highfreq_aggr2$Primer_pulse,
  levels = c(0, 5, 50, 500),
  labels = c("No pre-pulse", "Low pre-pulse",
             "Intermediate pre-pulse", "High pre-pulse")
)

highfreq_aggr2$Evo <- factor(
  highfreq_aggr2$Evo,
  levels = c("All ancestral","Pre-exposed 1977",
             "Pre-exposed 1972","All pre-exposed"),
  labels = c("All strains\nancestral",
             "Subdominant strain\npre-exposed",
             "Dominant strain\npre-exposed",
             "All strains\npre-exposed")
)

#### 14. Construct presence–absence table and run PERMANOVA
##### Create gene-by-population matrix and test effects of primer pulse and Evo on mutation profiles

In [14]:
gene_table0 <- highfreq_aggr[highfreq_aggr$evo_presence == "no", -c(1, 5)] %>%
  expand(Variantgene, Primer_pulse, Evo, Replicate)

gene_table1 <- merge(
  highfreq_aggr[highfreq_aggr$evo_presence == "no", -c(1, 5)],
  gene_table0,
  by  = c("Variantgene", "Primer_pulse", "Evo", "Replicate"),
  all = TRUE
)
gene_table1$COUNT[is.na(gene_table1$COUNT)] <- 0

gene_table2 <- dcast(
  gene_table1,
  Primer_pulse * Evo * Replicate ~ Variantgene,
  value.var = "COUNT",
  na.rm     = TRUE
)

permanova <- adonis2(
  gene_table2[, -c(1:3)] ~ Primer_pulse * Evo,
  data        = gene_table2[, 1:3],
  permutations = 9999,
  method       = "euc",
  na.rm        = TRUE,
  by           = "terms"
)

permanova <- cbind(
  "Factor" = rownames(as.data.frame(permanova)),
  as.data.frame(permanova)
)

permanova$Factor <- factor(
  permanova$Factor,
  levels = c("Primer_pulse", "Evo", "Primer_pulse:Evo", "Residual", "Total"),
  labels = c(
    "Pre-pulse",
    "Strain pre-exposure",
    "Pre-pulse x strain pre-exposure",
    "Residual",
    "Total"
  )
)

permanova

,Factor,Df,SumOfSqs,R2,F,Pr(>F)
,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Primer_pulse,Pre-pulse,1,2.649080,0.02887281,2.130726,0.0222
Evo,Strain pre-exposure,3,13.625000,0.14850136,3.652984,0.0001
Primer_pulse:Evo,Pre-pulse x strain pre-exposure,3,5.852482,0.06378727,1.569103,0.0309
Residual,Residual,56,69.623438,0.75883856,NA,NA
Total,Total,63,91.750000,1.00000000,NA,NA


#### 15. Pairwise PERMANOVA and beta-dispersion tests
##### Assess pairwise Evo differences and check multivariate dispersion among Evo groups

In [15]:
gene_distmat <- vegdist(gene_table2[, -c(1:3)], method = "euc")

pair_perm <- pairwise.adonis2(
  gene_distmat ~ Evo,
  data        = gene_table2[, 1:3],
  permutations = 999,
  by           = "terms"
)

pair_perm <- as.data.frame(pair_perm)
pair_perm <- cbind("Factor" = rownames(pair_perm), pair_perm)
pair_perm$Factor <- ifelse(pair_perm$Factor == "Evo",
                           "Strain pre-exposure", pair_perm$Factor)

pair_perm

dist <- vegdist(gene_table2[, -c(1:3)], method = "euc")

betadisp <- anova(betadisper(dist, gene_table2$Evo))

betadisp

,Factor,parent_call,All.ancestral_vs_All.pre.exposed.Df,All.ancestral_vs_All.pre.exposed.SumOfSqs,All.ancestral_vs_All.pre.exposed.R2,All.ancestral_vs_All.pre.exposed.F,All.ancestral_vs_All.pre.exposed.Pr..F.,All.ancestral_vs_Pre.exposed.1972.Df,All.ancestral_vs_Pre.exposed.1972.SumOfSqs,All.ancestral_vs_Pre.exposed.1972.R2,⋯,All.pre.exposed_vs_Pre.exposed.1977.Df,All.pre.exposed_vs_Pre.exposed.1977.SumOfSqs,All.pre.exposed_vs_Pre.exposed.1977.R2,All.pre.exposed_vs_Pre.exposed.1977.F,All.pre.exposed_vs_Pre.exposed.1977.Pr..F.,Pre.exposed.1972_vs_Pre.exposed.1977.Df,Pre.exposed.1972_vs_Pre.exposed.1977.SumOfSqs,Pre.exposed.1972_vs_Pre.exposed.1977.R2,Pre.exposed.1972_vs_Pre.exposed.1977.F,Pre.exposed.1972_vs_Pre.exposed.1977.Pr..F.
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Evo,Strain pre-exposure,"gene_distmat ~ Evo , strata = Null , permutations 999",1,8.4375,0.1898734,7.03125,0.001,1,1.625,0.03475936,⋯,1,7.25,0.1801242,6.590909,0.001,1,1.000,0.02318841,0.7121662,0.793
Residual,Residual,"gene_distmat ~ Evo , strata = Null , permutations 999",30,36.0000,0.8101266,NA,NA,30,45.125,0.96524064,⋯,30,33.00,0.8198758,NA,NA,30,42.125,0.97681159,NA,NA
Total,Total,"gene_distmat ~ Evo , strata = Null , permutations 999",31,44.4375,1.0000000,NA,NA,31,46.750,1.00000000,⋯,31,40.25,1.0000000,NA,NA,31,43.125,1.00000000,NA,NA


,Df,Sum Sq,Mean Sq,F value,Pr(>F)
,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Groups,3,0.6223405,0.2074468,1.09257,0.3592758
Residuals,60,11.3922312,0.1898705,NA,NA


#### 16. De novo mutation counts and Poisson regression
##### Summarize mutation counts per population and fit Poisson GLM for treatment effects

In [16]:
denovo_mut <- aggregate(
  COUNT ~ Primer_pulse * Evo * Replicate,
  FUN  = sum,
  data = highfreq_aggr[highfreq_aggr$evo_presence == "no", ]
)

denovo_mut2 <- denovo_mut %>%
  expand(Primer_pulse, Evo, Replicate)

denovo_mut3 <- merge(
  denovo_mut,
  denovo_mut2,
  by  = c("Primer_pulse", "Evo", "Replicate"),
  all = TRUE
)
denovo_mut3$COUNT[is.na(denovo_mut3$COUNT)] <- 0
head(denovo_mut3)

aov_mutcount <- as.data.frame(
  anova(glm(COUNT ~ Primer_pulse * Evo,
            data   = denovo_mut3,
            family = "poisson"))
)
rownames(aov_mutcount) <- c(
  "NULL",
  "Primer pulse",
  "Strain pre-exposure",
  "Primer pulse x strain pre-exposure"
)
aov_mutcount <- cbind("Factor" = rownames(aov_mutcount), aov_mutcount)

aov_mutcount

,Primer_pulse,Evo,Replicate,COUNT
,<dbl>,<chr>,<dbl>,<dbl>
1,0,All ancestral,1,0
2,0,All ancestral,2,2
3,0,All ancestral,3,0
4,0,All ancestral,4,1
5,0,All pre-exposed,1,2
6,0,All pre-exposed,2,2


,Factor,Df,Deviance,Resid. Df,Resid. Dev,Pr(>Chi)
,<chr>,<int>,<dbl>,<int>,<dbl>,<dbl>
NULL,NULL,NA,NA,63,49.48956,NA
Primer pulse,Primer pulse,1,3.724348,62,45.76521,0.05362456
Strain pre-exposure,Strain pre-exposure,3,2.184908,59,43.58031,0.53492722
Primer pulse x strain pre-exposure,Primer pulse x strain pre-exposure,3,3.083416,56,40.49689,0.37894206


#### 17. Confidence intervals and pre-pulse-specific genes
##### Compute mean mutation counts with CIs and identify genes only seen with pre-pulse > 0

In [17]:
ci_tbl <- denovo_mut3 %>%
  group_by(Primer_pulse) %>%
  summarise(
    mean_count = mean(COUNT),
    sd_count   = sd(COUNT),
    n          = n(),
    se         = sd_count / sqrt(n),
    ci_lower   = mean_count - qt(0.975, df = n - 1) * se,
    ci_upper   = mean_count + qt(0.975, df = n - 1) * se
  )

ci_tbl

aggregate(COUNT ~ Primer_pulse, FUN = mean, data = denovo_mut3)

pre_pulse <- as.data.frame.matrix(
  table(
    highfreq_aggr$Variantgene[highfreq_aggr$evo_presence == "no"],
    by = highfreq_aggr$Primer_pulse[highfreq_aggr$evo_presence == "no"]
  )
)
colnames(pre_pulse) <- c("No", "Low", "Intermediate", "High")

pre_pulse_specific <- pre_pulse[pre_pulse$No == 0, ]
pre_pulse_specific

# genes with mutations only with pre-pulse > 0
# purL: https://pmc.ncbi.nlm.nih.gov/articles/PMC9973305/ (only high pre-pulse)
# pfkA: https://pmc.ncbi.nlm.nih.gov/articles/PMC5473688/ (only high pre-pulse)
# yadV: https://pmc.ncbi.nlm.nih.gov/articles/PMC8409726/ (more with intermediate or high pre-pulse)
# pglF: https://www.mdpi.com/2079-6382/12/11/1619 (more with intermediate or high pre-pulse)

Primer_pulse,mean_count,sd_count,n,se,ci_lower,ci_upper
<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>
0,1.0625,0.9287088,16,0.2321772,0.567626,1.557374
5,1.6875,1.0781929,16,0.2695482,1.112972,2.262028
50,1.8750,0.8062258,16,0.2015564,1.445393,2.304607
500,2.2500,0.9309493,16,0.2327373,1.753932,2.746068


Primer_pulse,COUNT
<dbl>,<dbl>
0,1.0625
5,1.6875
50,1.8750
500,2.2500


,No,Low,Intermediate,High
,<int>,<int>,<int>,<int>
H1287_00097_rfaQ,0,0,0,1
H1287_01141_purL,0,0,0,2
H1287_01587_NA,0,0,1,1
H1287_01589_NA,0,0,0,1
H1287_02503_rne,0,0,0,1
H1287_02663_NA,0,0,0,1
H1287_02666_NA,0,0,0,1
H1896_00205_pfkA_1,0,0,1,1
H1896_01034_NA,0,0,0,2


#### 18. Export curated metagenome non-synonymous mutation table
##### Save filtered non-synonymous metagenome variants for downstream analyses

In [18]:
write_tsv(
  df_metagen6,
  "../data/mutation_metagen_ns.tsv"
)